# Project Final Report

In [2]:
### Run this cell before continuing.
import altair as alt
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.compose import make_column_transformer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Simplify working with large datasets in Altair
alt.data_transformers.enable('vegafusion')

# Output dataframes instead of arrays
set_config(transform_output="pandas")

np.random.seed(155)

## 1) Introduction

This project presents a unique opportunity to engage in a real-world data science initiative with active stakeholders seeking actionable insights from their data. The research focuses on understanding how individuals interact with a custom MineCraft server, with the broader goal of informing recruitment strategies and resource allocation for ongoing studies.

In this project, our group will address the following question: "Based on the number of hours played and the experience of the player, are they more likely to subscribe to the newsletter?" 

The players dataset below includes all variables required for this analysis. The explanatory variables will be “played_hours” (total hours played by each player) and “experience” (the player's level of experience). The response variable of interest is “subscribe,” which indicates whether a player has subscribed to the game-related newsletter. By analyzing the relationship between hours played and player experience, we explore whether we can predict that more/less engaged, experienced players will subscribe to the newsletter. 

#### (a) Loading and summarizing the data

In [8]:
sessions_data = pd.read_csv("./data/sessions.csv")
players_data = pd.read_csv("./data/players.csv")
players_data

,experience,subscribe,hashedEmail,played_hours,name,gender,age,individualId,organizationName
0,Pro,True,f6daba428a5e19a3d47574858c13550499be23603422e6...,30.3,Morgan,Male,9,NaN,NaN
1,Veteran,True,f3c813577c458ba0dfef80996f8f32c93b6e8af1fa9397...,3.8,Christian,Male,17,NaN,NaN
2,Veteran,False,b674dd7ee0d24096d1c019615ce4d12b20fcbff12d79d3...,0.0,Blake,Male,17,NaN,NaN
3,Amateur,True,23fe711e0e3b77f1da7aa221ab1192afe21648d47d2b4f...,0.7,Flora,Female,21,NaN,NaN
4,Regular,True,7dc01f10bf20671ecfccdac23812b1b415acd42c2147cb...,0.1,Kylie,Male,21,NaN,NaN
...,...,...,...,...,...,...,...,...,...
191,Amateur,True,b6e9e593b9ec51c5e335457341c324c34a2239531e1890...,0.0,Bailey,Female,17,NaN,NaN
192,Veteran,False,71453e425f07d10da4fa2b349c83e73ccdf0fb3312f778...,0.3,Pascal,Male,22,NaN,NaN
193,Amateur,False,d572f391d452b76ea2d7e5e53a3d38bfd7499c7399db29...,0.0,Dylan,Prefer not to say,17,NaN,NaN
194,Amateur,False,f19e136ddde68f365afc860c725ccff54307dedd13968e...,2.3,Harlow,Male,17,NaN,NaN


In [4]:
players_data_missing = players_data.isna().sum()
players_data_missing # shows that all rows of "individualId" and "organizationName" columns are missing

experience            0
subscribe             0
hashedEmail           0
played_hours          0
name                  0
gender                0
age                   0
individualId        196
organizationName    196
dtype: int64

The players data shown above contains the following 197 observations (rows) and 9 variables (columns), including:
1. experience: Categorical variable containing the player’s experience level.
2. subscribe: Categorical variable with whether player is subscribed or not subscribed to a game-related newsletter.
3. hashedEmail: Nominal variable that uniquely identifies each player.
4. played_hours: Numeric variable with total hours spent on Mine Craft server.
5. name: Nominal variable with the player’s name.
6. gender: Nominal variable with the player’s gender.
7. age: Discrete numeric variable with the player’s age.
8. individualId: NO DATA.
9. organizationName: NO DATA.

Issue:
- As shown in the players_data_missing, all row of the columns "individualId" and "organizationName" are missing data.

## (2) Methods and results

#### (a) Tidying the data

To prepare the data for analysis, we will first select only the relevant columns: “played_hours,” “experience,” and “subscribe.” Any unnecessary columns will be removed to simplify the modeling process. The predictive models we have encountered thus far operate using numerical values and distances. Experience is a categorical variable, but the categories correspond to the players' experience and have a clear and meaningful sequence, with 'beginner' as the least experienced and 'pro' as the most professional. Therefore, I will assign each experience level a number, ranging from 1 to 5. 

In [9]:
sessions_data["start_time"] = pd.to_datetime(sessions_data["start_time"])
sessions_data["end_time"] = pd.to_datetime(sessions_data["end_time"])
players_data_tidy = players_data.drop(["played_hours"], axis=1)
sessions_data.dropna(subset=['end_time'], inplace=True)
sessions_data['duration_hours'] = (sessions_data['end_time'] - sessions_data['start_time']).dt.total_seconds() / 3600
player_hours = sessions_data.groupby('hashedEmail')['duration_hours'].sum().reset_index()
player_hours.rename(columns={'duration_hours': 'played_hours'}, inplace=True)

players_data_tidy = pd.merge(players_data_tidy, player_hours, on='hashedEmail', how='left')
players_data_tidy['played_hours'] = players_data_tidy['played_hours'].fillna(0)

# Re-label subscribe "True" as "subscribed", and subscribe "False" as "not subscribed"
players_data_tidy['subscribe'] = players_data_tidy['subscribe'].replace({
    True : "subscribed",
    False : "not subscribed"
})
# Re-label experience levels with numbers
players_data_tidy['experience'] = players_data_tidy['experience'].replace({
    'Beginner' : 1,
    'Amateur' : 2,
    'Regular' : 3,
    'Veteran' : 4,
    'Pro' : 5
})

# Drop the columns identified as not useful
players_data_tidy.drop(columns=['individualId', 'organizationName', 'name'], inplace=True)

players_data_tidy


/var/folders/fm/9yjbfbfn4_xc8bw27rn0jdmr0000gn/T/ipykernel_62437/3147976809.py:1: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  sessions_data["start_time"] = pd.to_datetime(sessions_data["start_time"])
/var/folders/fm/9yjbfbfn4_xc8bw27rn0jdmr0000gn/T/ipykernel_62437/3147976809.py:2: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  sessions_data["end_time"] = pd.to_datetime(sessions_data["end_time"])
/var/folders/fm/9yjbfbfn4_xc8bw27rn0jdmr0000gn/T/ipykernel_62437/3147976809.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', Tr

,experience,subscribe,hashedEmail,gender,age,played_hours
0,5,subscribed,f6daba428a5e19a3d47574858c13550499be23603422e6...,Male,9,33.650000
1,4,subscribed,f3c813577c458ba0dfef80996f8f32c93b6e8af1fa9397...,Male,17,4.250000
2,4,not subscribed,b674dd7ee0d24096d1c019615ce4d12b20fcbff12d79d3...,Male,17,0.083333
3,2,subscribed,23fe711e0e3b77f1da7aa221ab1192afe21648d47d2b4f...,Female,21,0.833333
4,3,subscribed,7dc01f10bf20671ecfccdac23812b1b415acd42c2147cb...,Male,21,0.150000
...,...,...,...,...,...,...
191,2,subscribed,b6e9e593b9ec51c5e335457341c324c34a2239531e1890...,Female,17,0.000000
192,4,not subscribed,71453e425f07d10da4fa2b349c83e73ccdf0fb3312f778...,Male,22,0.350000
193,2,not subscribed,d572f391d452b76ea2d7e5e53a3d38bfd7499c7399db29...,Prefer not to say,17,0.083333
194,2,not subscribed,f19e136ddde68f365afc860c725ccff54307dedd13968e...,Male,17,2.983333


#### (b) Exploring and visualizing the data

In [29]:
# Count the number of occurrences of each value in subscribe.
players_data_tidy["subscribe"].value_counts()

subscribe
subscribed        144
not subscribed     52
Name: count, dtype: int64

In [10]:
players_plot = alt.Chart(players_data_tidy).mark_bar().encode(
    x=alt.X("played_hours").title("Play time in hours"),
    y=alt.Y("count()").title("Number of players"),
    color = alt.Color("subscribe").title("Subscription status"),
).facet(
    "subscribe:N",
    columns = 2,
    title = ("Figure 1.", "Number of players subscribed based on play time in hours")
)

players_plot

alt.FacetChart(...)

In [31]:
subscription_proportion = players_data.groupby('experience')['subscribe'].mean()
subscription_proportion_df = pd.DataFrame(subscription_proportion).reset_index()
experience_percent_plot = alt.Chart(subscription_proportion_df, title=["Figure 2.", "Percentage of players subscribed", "from each expereince level"]).mark_bar().encode(
    x=alt.X("experience").title("Player experience level"),
    y=alt.Y('subscribe').title("Percentage of players subscribed"),
)
experience_percent_plot

alt.Chart(...)

In [32]:
predictors_plot = alt.Chart(players_data, title=("Figure 3.", "Relationship between experience level and hours played")).mark_bar().encode(
    x=alt.X("played_hours").title("Total play time in hours"),
    y=alt.Y("experience").title("Player experience level"),
    color=alt.Color("experience").title("Player experience level")
)
predictors_plot

alt.Chart(...)

First, we visually explored the relationship between each predictor variable and our outcome of interest: subscription status. The first plot is a histogram of total hours played, separated by subscription status. This visualization reveals that, beyond a certain threshold of hours played, all players are subscribed to the newsletter. This pattern suggests that higher engagement, as measured by playtime, is associated with the likelihood of subscription. The plot highlights a clear cutoff point, reinforcing the idea that players who invest more time in the game are more likely to engage further by subscribing.

Next, we examined the percentage of players from each experience level who subscribed to the newsletter using a bar chart. This second visualization demonstrates that the subscription rate varies across experience categories. Notably, regulars have the highest subscription rate, while veterans and other groups show lower rates. This suggests that players at the 'regular' level might be at an optimal stage of engagement for subscribing. The variation across experience levels makes experience a relevant explanatory variable, and suggests that the relationship between experience and subscription may not be strictly linear.

The final visualization investigates the relationship between our two predictor variables—hours played and experience level—using a bar graph of play time by experience group. This analysis is important since our predictive methods rely on distance calculations, and it’s crucial to know if the predictors are correlated. The plot indicates that amateurs and regulars have the highest total play hours, while other groups contribute less overall playtime. Despite some relationship between experience and hours played, there remains meaningful variation within each experience group. This suggests that both predictors provide distinct information for modeling subscription status, reducing concerns about redundancy and supporting the inclusion of both features in our analysis.

Together, these exploratory visualizations offer important insights: they confirm that both the number of hours played and experience level are associated with newsletter subscription, and they clarify how these predictors relate to one another. This understanding supports our modeling choices and gives us greater confidence that our analysis will effectively capture the factors influencing subscription behavior


#### (c) Preparing the model

The method we used for this problem is K-nearest neighbors (KNN) classification. KNN classification is well-suited for predicting categorical outcomes. In this case, our target variable—newsletter subscription—has two possible values: subscribe (TRUE) or not subscribe (FALSE). This makes KNN a strong candidate for addressing our binary classification task.

KNN classification does not require assumptions about the underlying distribution or linearity of the data, which is advantageous here since we do not know the precise relationship between experience, hours played, and newsletter subscription. The primary assumption underlying KNN classification is that individuals with similar predictor values—here, experience and hours played—will have similar outcomes regarding newsletter subscription. The model assumes that these features are strong predictors of the decision to subscribe.

However, KNN has several potential limitations. It can become computationally slow with large datasets or a large number of predictors. Since the classes are slightly imbalanced (144 subscribers and 52 non-subscribers), the model may favor the majority class, especially if the K value is small. Additionally, because KNN is a distance-based method, it is essential to standardize predictor variables before applying the model, ensuring that no single variable biases predictions by disproportionately influencing the distance calculation.


To begin preparing our model, we split the dataset into a training set and a test set—using an 80/20 split. This ensures that we can train the model on one portion of the data and evaluate its performance on unseen data, providing a realistic assessment of predictive accuracy and minimizing the risk of overfitting. We then standardize the “played_hours” and “experience” variables, since they are measured on different scales. Standardization ensures that both variables contribute equally to KNN distance calculations and prevents any variable from disproportionately influencing the model’s predictions."

In [33]:
# Split stratify data and to ensure that the training and testing subsets 
# contain the right proportions of each category of observation.
players_train, players_test = train_test_split(
    players_data_tidy, train_size=0.8, stratify=players_data_tidy["subscribe"]
)

In [34]:
players_train.head()

,experience,subscribe,played_hours
50,4,subscribed,0.6
13,2,subscribed,0.2
93,2,subscribed,0.1
120,5,subscribed,0.1
163,3,subscribed,0.5


In [35]:
players_test.head()

,experience,subscribe,played_hours
4,3,subscribed,0.1
79,2,subscribed,0.0
69,3,subscribed,0.0
23,1,subscribed,0.0
18,2,subscribed,0.5


We can see from the info method above that the training set contains 156 observations, while the test set contains 40 observations. This corresponds to the desired train/test split of 80/20.

In [36]:
# Create a pipline for KNN classification
players_preprocessor = make_column_transformer(
    (StandardScaler(), ["experience", "played_hours"]),
)
knn = KNeighborsClassifier(n_neighbors=3)

X = players_train[["experience", "played_hours"]]
y = players_train["subscribe"]
X_test = players_test[["experience", "played_hours"]]
y_test = players_test["subscribe"]

knn_pipeline = make_pipeline(players_preprocessor, knn)
knn_pipeline.fit(X, y)

knn_pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('standardscaler',
                                                  StandardScaler(),
                                                  ['experience',
                                                   'played_hours'])])),
                ('kneighborsclassifier', KNeighborsClassifier(n_neighbors=3))])

With the pipeline complete, all necessary wrangling has been performed. 

#### (d) Buidling the model

Most predictive models in statistics and machine learning require the selection of key parameters. In KNN classification, the number of neighbors (K) is a crucial parameter that determines how many neighbors contribute to the class vote. By varying K, we can create different classifiers with varying predictive performance. Having already split our data into training and test sets and built an initial model with K=3, we next focused on finding the K value that yielded the highest accuracy. To avoid overfitting, we did not use the test set during model selection. Instead, we used cross-validation within the training data to evaluate model performance for different K values.

To ensure that every observation is used for both training and validation, we applied k-fold cross-validation, splitting the training data into evenly sized folds. Each fold was used once as a validation set, while the remaining folds were used for training. We first performed 5-fold cross-validation, then experimented with 10-fold cross-validation, which slightly reduced the standard error of our accuracy estimates. Based on these results, we opted for 10-fold cross-validation. This process yielded an estimated accuracy of around 64%. We used cross-validation accuracy to compare different K values and selected the K that maximized accuracy.


In [37]:
cv_5_df = pd.DataFrame(
    cross_validate(
        estimator=knn_pipeline,
        cv=5,
        X=X,
        y=y
    )
)

cv_5_df

,fit_time,score_time,test_score
0,0.005212,0.005313,0.718750
1,0.020240,0.004787,0.612903
2,0.004245,0.004550,0.580645
3,0.004218,0.005904,0.612903
4,0.005079,0.004795,0.709677


In [38]:
cv_5_metrics = cv_5_df.agg(["mean", "sem"])
cv_5_metrics

,fit_time,score_time,test_score
mean,0.007799,0.005070,0.646976
sem,0.003117,0.000243,0.028111


In [39]:
cv_10 = pd.DataFrame(
    cross_validate(
        estimator=knn_pipeline,
        cv=10,
        X=X,
        y=y
    )
)

cv_10_df = pd.DataFrame(cv_10)
cv_10_metrics = cv_10_df.agg(["mean", "sem"])
cv_10_metrics

,fit_time,score_time,test_score
mean,0.005404,0.004322,0.629167
sem,0.000873,0.000156,0.028171


In [40]:
cv_50_df = pd.DataFrame(
    cross_validate(
        estimator=knn_pipeline,
        cv=50,
        X=X,
        y=y
    )
)
cv_50_metrics = cv_50_df.agg(["mean", "sem"])
cv_50_metrics

/opt/conda/lib/python3.11/site-packages/sklearn/model_selection/_split.py:776: UserWarning: The least populated class in y has only 41 members, which is less than n_splits=50.
  warnings.warn(


,fit_time,score_time,test_score
mean,0.004344,0.003733,0.636667
sem,0.000085,0.000057,0.030094


To automate hyperparameter tuning, we used scikit-learn's GridSearchCV. We first created a pipeline with a KNeighborsClassifier, leaving the number of neighbors (n_neighbors) parameter empty so it could be optimized. We then defined a grid of possible values for n_neighbors and constructed a parameter_grid dictionary to instruct GridSearchCV which values to evaluate.

We constructed the GridSearchCV object by passing our pipeline as the estimator, the parameter_grid as the param_grid, and specifying 10-fold cross-validation (cv=10). We used the fit method on the GridSearchCV object, providing the training predictors and labels. The cv_results_ attribute contained cross-validation accuracy estimates for each n_neighbors value, which we converted into a pandas DataFrame for easier analysis.

In [41]:
knn_0 = KNeighborsClassifier()
players_tune_pipe = make_pipeline(players_preprocessor, knn_0)
parameter_grid = {
    "kneighborsclassifier__n_neighbors": range(1, 100, 5),
}
players_tune_grid = GridSearchCV(
    estimator=players_tune_pipe,
    param_grid=parameter_grid,
    cv=10
)
players_tune_grid.fit(
    X,
    y
)
accuracies_grid = pd.DataFrame(players_tune_grid.cv_results_)
accuracies_grid.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_kneighborsclassifier__n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
0,0.004695,0.000625,0.004324,0.000479,1,{'kneighborsclassifier__n_neighbors': 1},0.7500,0.6250,0.625,0.5625,0.6875,0.6250,0.666667,0.400000,0.666667,0.733333,0.634167,0.094038,19
1,0.004214,0.000062,0.004032,0.000062,6,{'kneighborsclassifier__n_neighbors': 6},0.6875,0.5625,0.625,0.5625,0.5625,0.6250,0.733333,0.600000,0.600000,0.600000,0.615833,0.053288,20
2,0.004260,0.000213,0.004305,0.000497,11,{'kneighborsclassifier__n_neighbors': 11},0.7500,0.7500,0.750,0.7500,0.7500,0.6875,0.733333,0.733333,0.733333,0.733333,0.737083,0.018300,1
3,0.004341,0.000407,0.004016,0.000057,16,{'kneighborsclassifier__n_neighbors': 16},0.6250,0.7500,0.750,0.7500,0.7500,0.6875,0.733333,0.600000,0.733333,0.600000,0.697917,0.061612,18
4,0.004304,0.000260,0.004080,0.000149,21,{'kneighborsclassifier__n_neighbors': 21},0.7500,0.7500,0.750,0.7500,0.7500,0.6875,0.733333,0.733333,0.733333,0.733333,0.737083,0.018300,1


From the accuracy grid, we isolated the relevant quantities: the number of neighbors (param_kneighbors_classifier__n_neighbors), the cross-validation accuracy estimate (mean_test_score), and the standard error of the accuracy estimate. Since GridSearchCV outputs the standard deviation (std_test_score) rather than the standard error, we computed the standard error by dividing the standard deviation by the square root of the number of folds.

In [42]:
accuracies_grid["sem_test_score"] = accuracies_grid["std_test_score"] / 10**(1/2)
accuracies_grid = (
    accuracies_grid[[
        "param_kneighborsclassifier__n_neighbors",
        "mean_test_score",
        "sem_test_score"
    ]]
    .rename(columns={"param_kneighborsclassifier__n_neighbors": "n_neighbors"})
)
accuracies_grid

,n_neighbors,mean_test_score,sem_test_score
0,1,0.634167,0.029737
1,6,0.615833,0.016851
2,11,0.737083,0.005787
3,16,0.697917,0.019483
4,21,0.737083,0.005787
5,26,0.737083,0.005787
6,31,0.737083,0.005787
7,36,0.737083,0.005787
8,41,0.737083,0.005787
9,46,0.737083,0.005787


In [43]:
accuracy_vs_k = alt.Chart(accuracies_grid, title=("Figure 4.", "Estimated accuracy versus the number of neighbors")).mark_line(point=True).encode(
    x=alt.X("n_neighbors").title("Neighbors"),
    y=alt.Y("mean_test_score")
        .scale(zero=False)
        .title("Accuracy estimate")
)

accuracy_vs_k

alt.Chart(...)

In [44]:
players_tune_grid.best_params_

{'kneighborsclassifier__n_neighbors': 11}

To identify the optimal number of neighbors, we plotted accuracy versus K and also programmatically accessed the best_params_ attribute of the fitted GridSearchCV object. Both methods indicated that accuracy increased rapidly with K and then plateaued, helping us pinpoint the optimal K. Setting n_neighbors to 11 yielded the highest cross-validation accuracy estimate.

In [45]:
players_test["predicted"] = players_tune_grid.predict(
    players_test[["experience", "played_hours"]]
)

players_tune_grid.score(
    players_test[["experience", "played_hours"]],
    players_test["subscribe"]
)

0.725

After finalizing the model, we evaluated its predictive performance on the held-out test set. We retrained the KNN classifier on the full training data using the chosen number of neighbors (which scikit-learn's GridSearchCV handles automatically). We then used the score and predict methods of the fitted GridSearchCV object to assess the model's accuracy and generate predictions on the test data.

The final model achieved an accuracy of 72.5% on the test set.

In [46]:
pd.crosstab(
    players_test["subscribe"],
    players_test["predicted"]
)

predicted,subscribed
subscribe,
not subscribed,11
subscribed,29


From the crosstab we can see that the classifier only ever predicted that players would subscribe to the newsletter.

## 3) Discussion

This analysis set out to determine whether two key features—experience points and played hours—could accurately predict whether a player would subscribe to the game-related newsletter. Our expectation, based on common intuition, was that players who invested more time and had greater experience would be more likely to subscribe, reflecting stronger engagement or commitment to the game.

However, the results did not reveal what we expected. Despite our initial exploration of the variables used and thorough hyperparameter tuning, the predictive K-Nearest Neighbors (KNN) classification model performed no better than a majority-class baseline. In other words, experience and played hours did not distinguish subscribers from non-subscribers in this dataset.

One possible explanation for this unexpected outcome lies in the data source: the Minecraft server was primarily used by college students as part of an academic assignment. This context limited the diversity of player behavior, since many students may have engaged just enough to satisfy course requirements rather than out of genuine interest. As seen in our exploratory visualizations, there were also a few outliers who contributed a large number of hours in comparison to the other students. Consequently, both the predictors (experience and played hours) and the outcome (subscribe) showed little variation, making it difficult for any model to detect meaningful patterns or achieve accurate predictions. The limited variability in both features and the target variable, driven by the specific academic collection context, likely prevented the KNN model from finding any meaningful, generalizable differences between subscribers and non-subscribers.

The impact of these findings is significant for the research group and stakeholders. First, simple behavioral metrics such as playtime or experience cannot be relied on in this context to identify or target committed players for future studies or resource planning. Also, forecasting server needs or licenses based on these features risks inefficient allocation, as there is no reliable way to segment players by expected engagement using current variables. The results highlight the need for more nuanced, context-specific behavioral data to understand better and predict player engagement or subscription decisions.

These findings raise several important questions for the future:
- What additional data (qualitative feedback, motivation surveys, in-game activity) could provide a more accurate picture of engagement?
- How might player behavior differ in a less specific, more natural environment?
- Would alternative modeling approaches have different results, or is the lack of predictive power due in its entirety to the current context?

In summary, while the initial hypothesis was not supported, these results provide valuable guidance for refining research methods and data collection strategies in future work.
